In [1]:
import torch

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cpu'

In [16]:
data = open('./names.txt', mode='r').read()

In [17]:
vocab = sorted(list(set(data.lower())))
print(vocab)

['\n', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [18]:
char_to_id = {ch: id for id, ch in enumerate(vocab)}
id_to_char = {id: ch for id, ch in enumerate(vocab)}
char_to_id['a']

1

In [19]:
def encode_text(text):
    return torch.tensor([char_to_id[ch] for ch in text.lower()])

def decode_text(token_ids):
    return ''.join([id_to_char[ch.item()] for ch in token_ids])

In [20]:
encoded = encode_text('khalid')
print(encoded)
decode_text(encoded)

tensor([11,  8,  1, 12,  9,  4])


'khalid'

In [21]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length


    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset is out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx + 1 : end + 1]
        return window, target


In [23]:
my_dataset = CharDataset(text="khalid", window_length=3)
for window, target in my_dataset:
    print("Window:", window, " Target:", target)


Window: tensor([11,  8,  1])  Target: tensor([ 8,  1, 12])
Window: tensor([ 8,  1, 12])  Target: tensor([ 1, 12,  9])
Window: tensor([ 1, 12,  9])  Target: tensor([12,  9,  4])


In [24]:
window_length = 50
batch_size = 512
train_set = CharDataset(data[:3_000], window_length)
valid_set = CharDataset(data[3_000:3393], window_length)
test_set = CharDataset(data[3393:], window_length)
train_loader = DataLoader(train_set, batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size)

In [25]:
for x, y in train_loader:
    print('x', x.shape)
    print('y', y.shape)
    break

x torch.Size([512, 50])
y torch.Size([512, 50])


In [26]:
# Embeddings
import torch.nn as nn
torch.manual_seed(42)
embd = nn.Embedding(5, 3) # 5 categories and 3D embeddings
embd(torch.tensor([[3,3, 2], [3,1,2]]))

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.2674,  0.5349,  0.8094],
         [ 0.2303, -1.1229, -0.1863],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

In [27]:
# equalient to this
ln = nn.Linear(5, 3, bias=False)
ln(torch.tensor([0., 0., 0., 0., 1]))

tensor([ 0.0697, -0.1213,  0.2582], grad_fn=<SqueezeBackward4>)

In [28]:
class ShakespareModel(nn.Module):
    def __init__(self, vocab_size, n_layers = 2, embed_dim = 10, hidden_dim = 128, dropout = 0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout
        )
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)


torch.manual_seed(42)
ssm = ShakespareModel(len(vocab)).to(device)


In [29]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

In [30]:
n_epochs = 20
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(ssm.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocab)).to(device)
history = train(ssm, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs)

Epoch 1/20,                      train loss: 3.0998, train metric: 13.42%, valid metric: 15.01%
Epoch 2/20,                      train loss: 2.7372, train metric: 19.97%, valid metric: 19.87%
Epoch 3/20,                      train loss: 2.6405, train metric: 21.20%, valid metric: 22.35%
Epoch 4/20,                      train loss: 2.5324, train metric: 23.93%, valid metric: 23.33%
Epoch 5/20,                      train loss: 2.4471, train metric: 26.27%, valid metric: 25.41%
Epoch 6/20,                      train loss: 2.3884, train metric: 27.60%, valid metric: 26.80%
Epoch 7/20,                      train loss: 2.3373, train metric: 28.49%, valid metric: 26.17%
Epoch 8/20,                      train loss: 2.2870, train metric: 29.52%, valid metric: 25.83%
Epoch 9/20,                      train loss: 2.2594, train metric: 30.47%, valid metric: 26.71%
Epoch 10/20,                      train loss: 2.2000, train metric: 31.67%, valid metric: 27.90%
Epoch 11/20,                      train

In [35]:
ssm.eval()  # don't forget to switch the model to evaluation mode!
text = "em"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = ssm(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]  

In [36]:
predicted_char

'\n'

In [37]:
import torch.nn.functional as F

def next_char(model, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

In [ ]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        ch = next_char(model, text, temperature)
        if(ch=='\n'):
            break
        text+=ch
    return text

In [64]:
extend_text(ssm, "ol", temperature=0.3)

'olie\nsaulie\nelielle\namania\nmaria\nalana\ndarley\ngarison\naliana\nalia\ncadile\nleyna\nmar'

In [61]:
extend_text(ssm, "h", temperature=0.04)

'hia'

In [62]:
extend_text(ssm, "j", temperature=100)

'jlzanwglxjgxbegahmjotnsbwysq'

# Sentiment Analysis

In [ ]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")
split = imdb_dataset["train"].train_test_split(train_size = 0.8, seed=42)
imdb_train_set, imdb_valid_set = split['train'], split['test']
imdb_test_set = imdb_dataset['test']

In [ ]:
imdb_train_set[1]["text"], imdb_train_set[1]["label"]

In [ ]:
!pip install tokenizors

In [ ]:
import tokenizers

bpe_model = tokenizers.model.BPE(unk_token='<unk>')
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ['<pad>', '<unk>']
bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size = 1000,
    special_tokens = special_tokens
)
train_reviews = [review['train'].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)


In [ ]:
some_review = "this is good movie"
bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding

In [ ]:
bpe_encoding.tokens, bpe_encoding.ids

In [ ]:
bpe_encoding.decode(bpe_encoding.ids)

In [ ]:
bpe_encoding.offsets

In [ ]:
bpe_encoding.encode_batch(
    train_reviews[:3]
)

In [ ]:
# adding the padding
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [ ]:
bpe_encoding.encode_batch(
    train_reviews[:3]
)

In [ ]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encoding])


In [ ]:
import transformers
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True, max_length=500)

In [ ]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
gpt2_token_ids , gpt2_tokenizer.decode(gpt2_token_ids)

In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True,
 truncation=True, max_length=500,
 return_tensors="pt")

In [ ]:
# we can pass a function to the DataLoader constructor using its collate_fn argument: the data loader will call this function for every batch, passing it a list of dataset samples. 

def collate_fn(batch, tokenizer=bert_tokenizer):
    reivews = [ review['text'] for review in batch]
    labels = [review['label'] for review in batch]
    encodings = tokenizer(reivews , padding=True, truncation=True, max_length=200, return_tensors='pt')
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings, labels

batch_size = 256
imdb_train_loader = DataLoader(imdb_train_set, batch_size, collate_fn, shuffle=True)
imdb_test_loader = DataLoader(imdb_test_set, batch_size, collate_fn)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size, collate_fn)

In [ ]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64, pad_id=0,dropout=0.2):
        self.__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, batch_first=True, num_layers=n_layers, dropout=dropout,
        )
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])

In [ ]:
# Train the model
torch.manual_seed(42)
vocab_size = bert_tokenizer.vocab_size
imdb_model_ps = SentimentAnalysisModel(vocab_size).to(device)
n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_ps.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history = train(imdb_model_ps, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

One problem with our model is the fact that we are not fully ignoring the padding tokens.
Indeed, if a review ends with many padding tokens, the nn.GRU module will have to
process them, and by the time it gets through all of them, it might have forgotten what the
review was all about. To avoid this, we can use a packed sequence instead of a regular
tensor. A packed sequence is a special data structure designed to efficiently represent a
batch of sequences of variable lengths.12 You can use the pack_padded_sequence()
function to convert a tensor containing padded sequences to a packed sequence object,
and conversely you can use the pad_packed_sequence() function whenever you want to
convert a packed sequence object to a padded tensor:

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

sequences = torch.tensor([[1, 2, 0, 0], [5, 6, 7, 8]])
packed = pack_padded_sequence(sequences, lengths=(2, 4),
                              enforce_sorted=False, batch_first=True)
packed

In [ ]:
padded, lengths = pad_packed_sequence(packed, batch_first=True)
padded, lengths

In [ ]:
class SentimentAnalysisModelPackedSeq(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim,
                                  padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)                      # <= line added
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),      # <= line added
                                      batch_first=True, enforce_sorted=False) # <= line added
        _outputs, hidden_states = self.gru(packed)                            # <= line changed
        return self.output(hidden_states[-1])

A regular recurrent layer only looks at past and present inputs before
generating its output. In other words, it is causal, meaning it cannot look into the future.
This type of RNN makes sense when forecasting time series, or in the decoder of a
sequence-to-sequence (seq2seq) model. But for tasks like text classification, or in the
encoder of a seq2seq model, it is often preferable to look ahead at the next words before
encoding a given word.
For example, consider the phrases “the right arm”, “the right person”, and “the right to
speak”: to properly encode the word “right”, you need to look ahead. One solution is to
run two recurrent layers on the same inputs, one reading the words from left to right and
the other reading them from right to left, then combine their outputs at each time step,
typically by concatenating them. This is what a bidirectional recurrent layer does 

In [ ]:
class SentimentAnalysisModelBidi(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim,
                                  padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)  # <= line changed
        self.output = nn.Linear(2 * hidden_dim, 1)                               # <= line changed

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features                                          # <= line added
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)      # <= line added
        return self.output(top_states)                                            # <= line changed

In [ ]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_bidi = SentimentAnalysisModelBidi(vocab_size).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bidi.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bidi, optimizer, xentropy, accuracy, imdb_train_loader,
                imdb_valid_loader, n_epochs)